# Sistema de recomendação de hotéis — documentação do modelo

**Introdução à Inteligência Artificial — Trabalho 1**

**Integrantes**
- Artur Kohara Guerra — matrícula 231025181  
- Celio Junio de Freitas Eduardo — matrícula 211010350  
- Rafael De Lima Pereira — matrícula 242043277  

Este notebook resume a mesma modelagem descrita em `relatorio.tex`: dados sintéticos em `data_generator.py`, vetor de busca e dois modos de ordenação em `recomendacao_controller.py`. Execute as células **Python** na ordem (Python 3.10+; `pip install -r requirements.txt` no diretório do projeto para `numpy`, `pandas`, etc.).

**A interface web (Streamlit) não deve ser iniciada dentro do Jupyter** — ela roda num terminal à parte, os passos estão na próxima seção.

## Como executar o programa (interface Streamlit)

### Pré-requisitos
- Python **3.10 ou superior**
- Pasta do projeto no disco

### Passo a passo

1. **Abra um terminal** e vá para a pasta raiz do projeto (onde estão `app.py` e `requirements.txt`):
   ```bash
   cd caminho/para/iia-recomendation-t1
   ```

2. **(Recomendado)** Crie e ative um ambiente virtual para isolar dependências:
   ```bash
   python3 -m venv .venv
   source .venv/bin/activate          # Linux / macOS
   # ou:  .venv\Scripts\activate     # Windows (cmd/PowerShell)
   ```

3. **Instale as dependências** (Streamlit, pandas, etc.):
   ```bash
   python3 -m pip install --upgrade pip
   python3 -m pip install -r requirements.txt
   ```

4. **Inicie a aplicação**:
   ```bash
   streamlit run app.py
   ```

5. O Streamlit mostra um endereço local (por padrão **http://localhost:8501**). Abra no navegador; se não abrir sozinho, copie e cole o URL.

6. Na primeira execução, o código cria/atualiza o SQLite `sistema_recomendacao.db` e popula dados sintéticos se o banco estiver vazio (`ui_data.ensure_database_ready`).

### O que este notebook cobre
Análise reprodutível da **modelagem numérica** (vetores, cosseno, score linear). O fluxo completo com cadastro e méstricas fica na **interface Streamlit**, pelos passos acima.

## 1. Formulação do problema

- Cada hotel $i$ é $\mathbf{h}_i \in [0,1]^{10}$ (luxo … capacidade).
- Região filtra candidatos antes do score; não entra como componente numérica do cosseno.
- Avaliações explícitas $r_{ui} \in \{1,\ldots,5\}$ na matriz de utilidade.

Carregamos o catálogo gerado deterministicamente (`np.random.seed(42)` em `data_generator.py`).

In [ ]:
import os
import sys

# Raiz do repositório (ajuste se o notebook estiver em outra pasta)
ROOT = os.path.abspath(os.path.join(os.getcwd(), ""))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import pandas as pd
from data_generator import df_hotels, FEATURES, REGIOES

_df = df_hotels.rename(
    columns=lambda x: x.lower()
    .replace("petfriendly", "pet_friendly")
    .replace("kidsfriendly", "kids_friendly")
)
COLS = [c for c in _df.columns if c != "regiao"]

print("Hotéis:", len(df_hotels), "| Regiões:", REGIOES)
print("Colunas numéricas:", COLS)
df_hotels.head(3)

## 2. Vetor de busca: contexto, perfil e fusão

- $\mathbf{v}_{\mathrm{ctx}}$: regras do formulário (`_build_context_vector` no controlador).
- $\mathbf{v}_{\mathrm{perfil}}$: média ponderada das features dos hotéis avaliados, pesos = notas.
- $\mathbf{v} = \alpha \mathbf{v}_{\mathrm{ctx}} + (1-\alpha) \mathbf{v}_{\mathrm{perfil}}$ com $\alpha = 0{,}85$ se houver histórico.

Abaixo um exemplo **sintético** de perfil e contexto apenas para ilustrar a Eq. de fusão (sem depender do SQLite).

In [ ]:
ALPHA = 0.85
rng = np.random.default_rng(0)
v_ctx = rng.uniform(0.1, 1.0, size=len(COLS))
v_perfil = rng.uniform(0.2, 0.9, size=len(COLS))

v = ALPHA * v_ctx + (1 - ALPHA) * v_perfil
pd.Series(v, index=COLS)

## 3. Modo cosseno (rótulo *KNN* na interface)

$$
s_i^{(\cos)} = \frac{\mathbf{h}_i^\top \mathbf{v}}{\|\mathbf{h}_i\|\|\mathbf{v}\| + \varepsilon}
$$

Filtramos uma região e ordenamos por $s_i^{(\cos)}$ decrescente.

In [ ]:
EPS = 1e-9

def cosine_scores(df_region: pd.DataFrame, v: np.ndarray, feature_cols: list[str]) -> np.ndarray:
    H = df_region[feature_cols].values
    dot = H @ v
    norm_h = np.linalg.norm(H, axis=1)
    nv = np.linalg.norm(v)
    return dot / (norm_h * nv + EPS)

regiao = "Sao Paulo"
sub = _df[_df["regiao"] == regiao]
scores_cos = cosine_scores(sub, v, COLS)
rank = np.argsort(-scores_cos)
sub.iloc[rank[:5]].assign(score_cos=scores_cos[rank[:5]])

## 4. Modo linear + penalidade (rótulo *FM*)

**Não** é Factorization Machines treinada; é:
$$
s_i^{(\mathrm{lin})} = \mathbf{h}_i^\top \mathbf{v} - \lambda \, h_{i,\mathrm{luxo}} \, h_{i,\mathrm{urbano}}, \quad \lambda = 0{,}6
$$

In [ ]:
LAMBDA_PEN = 0.6

def linear_scores(df_region: pd.DataFrame, v: np.ndarray, feature_cols: list[str]) -> np.ndarray:
    H = df_region[feature_cols].values
    base = H @ v
    pen = LAMBDA_PEN * H[:, 0] * H[:, 2]  # luxo, urbano
    return base - pen

scores_lin = linear_scores(sub, v, COLS)
rank2 = np.argsort(-scores_lin)
sub.iloc[rank2[:5]].assign(score_linear=scores_lin[rank2[:5]])

## 5. Hiperparâmetros (reprodutibilidade)

| Símbolo | Valor | Origem |
|---------|-------|--------|
| $\alpha$ | 0,85 | `alpha_adaptacao` |
| $\lambda$ | 0,6 | `LAMBDA_PENALTY` / `_computar_fm` |
| $\varepsilon$ | $10^{-9}$ | cosseno |
| Paginação | 5 | `tamanho_pagina` |
| Seed | 42 | `data_generator.py` |

## 6. Métricas no código — limitações

- **Cobertura**: `ui_data.get_catalog_coverage` — hotéis distintos em `avaliacoes` / total em `hoteis`.
- **RMSE** no painel para rótulo FM usa `_obter_predicao_fm_mock` (valor fixo por hotel no protótipo), não erro de modelo treinado.
- **NDCG** simplificado: média de $1/\log_2(\mathrm{pos}+1)$ onde `logica_geracao == 'KNN'` (ver `recomendacao_controller._gerar_dashboard_metricas_globais`).

### Referências (mesmas do relatório)
- Pazzani & Billsus (2007), conteúdo.  
- Rendle (2010), FM (referência conceitual).  
- Järvelin & Kekäläinen (2002), avaliação por ganho.

---

**Demonstração com interface gráfica:** consulte a seção **Como executar o programa (interface Streamlit)** no início deste notebook.